In [ ]:
import kagglehub
from IPython.display import clear_output

%pip install kagglehub catboost lightgbm tqdm -q

clear_output()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

%matplotlib inline

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df,"Delivery_Time")

In [ ]:
# Task 1: Write your code here:
# Define stat columns
stat_cols = ['Order_ID']

# Drop rows with missing stat values
df = df.dropna(subset=stat_cols)
print(f"Shape after cleaning: {df.shape}")

In [ ]:
# Task 2: Write your code here:
# 2. Do we have missing values?
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

# 3. Do we have categorical columns?
def encode_categorical_columns(df):
    categorical_cols = df.select_dtypes(include=["object"]).columns
    print("Categorical Columns:", list(categorical_cols))

# Fill categorical columns with 'unknown' - missing likely means "not specified"
for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    df[col] = df[col].fillna('unknown')

# Fill Delivery_Time with mean
df['Delivery_Time'] = df['Delivery_Time'].fillna(df['Delivery_Time'].mean())

# Fill Delivery_Time with mean
df['Courier_Experience_yrs'] = df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].mean())

check_missing_values(df)

print("Missing values remaining:", df.isnull().sum().sum())

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:

from sklearn.preprocessing import OneHotEncoder

categorical_data = [['Weather'], ['Traffic_Level'], ['Time_of_Day'],['Vehicle_Type']]

print(f"Original Data:\n{categorical_data}\n")

# 2. Initialize the encoder (Ensure output is dense/numpy array)
my_encoder = OneHotEncoder(sparse_output = False)

# 3. Fit and Transform the data
encoded_data = my_encoder.fit_transform(categorical_data)

# 4. Print the result
print(encoded_data)

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

# TODO: Initialize StandardScaler
standard_scaler = StandardScaler()

# TODO: Apply fit_transform to df_exercise
df_exercise_scaled = standard_scaler.fit_transform(encoded_data)


print(f"data after scaling {encoded_data}")


In [ ]:
# Task 6: Write your code here:

# 1. Is the target imbalanced?
def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df[target_column].hist()  # Yeah you can just do this :)
  plt.show()

check_target_imbalance(df, "Delivery_Time")

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error as sklearn_mae
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1)
y = df['Delivery_Time']

In [ ]:

n_splits = 5 # K=5 Folds

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

from sklearn.model_selection import train_test_split

# CORRECTED: Use capital 'X' to match the previous definition
X, y = X.copy(), y.copy()

# split ratio (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% for test, remaining 80% for train
    random_state=42,      # reproducible output
    shuffle=True,         # representative splits
    stratify=y            # preserve class distribution
)

# Print shapes
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)



from sklearn.model_selection import KFold

X, y = X_reg.copy(), y_reg.copy()

kf = KFold(n_splits = 5, shuffle = True, random_state = 42)

for fold , (train_idx, test_idx) in enumerate(kf.split(X), start = 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    print(f"Fold {fold}")
    print("  X_train shape:", X_train.shape)
    print("  X_test shape :", X_test.shape)
    print("  y_train shape:", y_train.shape)
    print("  y_test shape :", y_test.shape)
    print("-" * 30)

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mse = sklearn_mse(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    # Store results
    all_results[model_name]["mse"].append(mse)
    all_results[model_name]["rmse"].append(rmse)
    all_results[model_name]["r2"].append(r2)

# Make predictions and calculate MAE
valid_pred = model.predict(X_valid)
model_mae = mean_absolute_error(y_valid, valid_pred)
print(f"Model MAE: {model_mae:.4f}")

# Calculate average loss across folds
avg_loss = np.mean(lr_losses, axis=0)

plt.figure(figsize=(10, 6))
plt.plot(avg_loss, label='Logistic Regression Loss', color='purple')
plt.title('Average Logistic Regression Loss Curve (Across 5 Folds)')
plt.xlabel('Iteration')
plt.ylabel('Binary Cross-Entropy Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Task 1: Write your code here:
# Gather importances from the models (from the last fold)
importances = {}

importances['Random Forest'] = sklearn_models['Random Forest'].feature_importances_

# Create a 1x3 plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 6))
plt.hist(valid_pred, bins=30, edgecolor='black')
plt.title('Distribution of Predictions')
plt.xlabel('Predicted Emission')
plt.ylabel('Count')
plt.show()



In [ ]:
# TODO: Define models with hyperparameters of your choice
models = {
  "Random Forest": RandomForestRegressor (
      n_estimators=200,
      max_depth=10
  ),
  "CatBoost": CatBoostRegressor(
      verbose=0,
      n_estimators=200,
      max_depth=4
  )
}

average_losses = np.mean(lr_losses, axis=0)

plt.figure(figsize=(10, 6))
plt.plot(average_losses, label='Average Loss')
plt.xlabel('Iteration')
plt.ylabel('MSE Loss')
plt.title('Linear Regression Training Loss (Averaged Across Folds)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

results = {}

for model_name in models:
  results[model_name] = {'accuracy': [], 'precision': [], 'recall': [], 'f1': []}

  for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # Get the train & test split for this fold
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train & Validate Models
  for model_name, model in models.items():

    print(f"Training {model_name}...")

    # Fit the model on train data
    model.fit(X_train, y_train)

    # Use the model to predict the test data
    y_pred = model.predict(X_test)

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')  # for multiclass f1 score, you should set the average hyperparameter ("macro", "micro", "weighted")

    results[model_name]['accuracy'].append(accuracy)
    results[model_name]['f1'].append(f1)